# 23 · Load: graduates (IPEDS completions + directory, CUNY Programs) → `staging`

**Source cards:** `ipeds_completions.md`, `cuny_programs.md` (§6 issues, §7 staging tables). Explored in: notebook 05.

| Table | One row = | Expected rows |
|---|---|---|
| `stg_ipeds__completions` | year × NYC college × major (detailed CIP) × major number × award level | 58,343 |
| `stg_ipeds__institutions` | year × NYC college | 1,836 |
| `stg_cuny__program_graduates` | CUNY program (IRP code × college × award) × academic year | 53,838 |
| `stg_cuny__program_enrollment` | CUNY program × fall term | 53,838 |

Hand-built maps (both in `ref/`, loaded later as dbt seeds): `cuny_unit_unitid.csv` (26 CUNY units → 20 IPEDS colleges; 6 are parent-college assumptions) and `cuny_award_level.csv` (CUNY award name → level).

Reading ten years of IPEDS takes about a minute.

In [1]:
import io
import sys
import zipfile
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
from common import log
from staging import connect, raw_files, stage

pd.set_option("display.max_columns", 40, "display.max_colwidth", 70, "display.width", 220)
NB = "23-load-graduates.ipynb"
REF = PROJECT / "ref"
NYC_COUNTIES = {"36005": "Bronx", "36047": "Brooklyn", "36061": "Manhattan", "36081": "Queens", "36085": "Staten Island"}
CUNY_SYSTEM = "136010"
conn = connect()
summary = []

def decode(raw: bytes) -> str:
    """IPEDS files are Latin-1 through 2022 and UTF-8 from 2023 (issue 3)."""
    try:
        return raw.decode("utf-8-sig")
    except UnicodeDecodeError:
        return raw.decode("latin-1")

def read_zip(path: Path, prefer_revised: bool, **kw) -> tuple[pd.DataFrame, str, str]:
    """The revised `_rv` member when there is one (issue 1), else the only CSV member."""
    z = zipfile.ZipFile(path)
    members = [m for m in z.namelist() if m.lower().endswith(".csv")]
    rv = [m for m in members if "_rv" in m.lower()]
    member = rv[0] if (prefer_revised and rv) else next(m for m in members if m not in rv)
    d = pd.read_csv(io.StringIO(decode(z.read(member))), dtype=str, keep_default_na=False, **kw)
    return d.apply(lambda s: s.str.strip()), member, "revised" if rv and prefer_revised else "provisional"

## 1. IPEDS directory (HD): which colleges are in NYC each year

NYC membership comes from each year's county code (issue 10: colleges close and move). HD text is space-padded from 2023 (issue 4), so every text column is stripped (done in `read_zip`), and CUNY is flagged on the system code `F1SYSCOD = 136010`, not the name.

In [2]:
HD_COLS = ["UNITID", "INSTNM", "CITY", "STABBR", "COUNTYCD", "SECTOR", "ICLEVEL", "CONTROL", "F1SYSNAM", "F1SYSCOD", "CLOSEDAT"]
SECTOR = {"0": "Administrative unit", "1": "Public 4-year+", "2": "Private nonprofit 4-year+", "3": "Private for-profit 4-year+",
          "4": "Public 2-year", "5": "Private nonprofit 2-year", "6": "Private for-profit 2-year",
          "7": "Public < 2-year", "8": "Private nonprofit < 2-year", "9": "Private for-profit < 2-year", "99": "Sector unknown (not active)"}
CONTROL = {"1": "Public", "2": "Private nonprofit", "3": "Private for-profit"}
LEVEL = {"0": "Administrative unit", "1": "Four or more years", "2": "At least 2 but less than 4 years", "3": "Less than 2 years"}

hd_parts = []
for f, path in zip(raw_files("ipeds_directory").file, raw_files("ipeds_directory").path):
    year = int(Path(f).stem[2:6])
    d, member, _ = read_zip(path, prefer_revised=False, usecols=HD_COLS)
    nyc = d[d.COUNTYCD.isin(NYC_COUNTIES)].assign(year=year, source_file=f"{f}!{member}")
    log(f"HD{year}: keep NYC counties", len(d), len(nyc))
    hd_parts.append(nyc)
hd = pd.concat(hd_parts, ignore_index=True)
for col, m in [("SECTOR", SECTOR), ("CONTROL", CONTROL), ("ICLEVEL", LEVEL)]:
    unknown = set(hd[col]) - set(m) - {"-3"}
    assert not unknown, f"{col}: unknown codes {unknown}"

inst = pd.DataFrame({
    "year": hd.year.astype("Int64"), "unitid": hd.UNITID, "name": hd.INSTNM, "city": hd.CITY,
    "county_fips": hd.COUNTYCD, "borough": hd.COUNTYCD.map(NYC_COUNTIES),
    "sector": hd.SECTOR.map(SECTOR), "control": hd.CONTROL.map(CONTROL), "level": hd.ICLEVEL.map(LEVEL),
    "is_cuny": hd.F1SYSCOD.eq(CUNY_SYSTEM), "system_name": hd.F1SYSNAM.replace({"-2": None, "": None}),
    "closed_date": hd.CLOSEDAT.replace({"-2": None, "": None}), "source_file": hd.source_file})
print("Code -3 (not applicable) cells mapped to NULL:", {c: int((hd[c] == "-3").sum()) for c in ["SECTOR", "CONTROL", "ICLEVEL"]})
display(pd.DataFrame({"NYC colleges": inst.groupby("year").size(), "CUNY units": inst[inst.is_cuny].groupby("year").size()}).T)
assert inst[inst.is_cuny].groupby("year").size().eq(21).all()
summary.append(stage(inst, "stg_ipeds__institutions", ["year", "unitid"], expected_rows=1_836,
                     source_key="ipeds_directory", notebook=NB, conn=conn,
                     comment="IPEDS HD, NYC colleges (5 borough counties) per year; is_cuny from F1SYSCOD = 136010."))

  HD2015: keep NYC counties: 7,647 -> 191 rows (-7,456)
  HD2016: keep NYC counties: 7,521 -> 191 rows (-7,330)
  HD2017: keep NYC counties: 7,153 -> 187 rows (-6,966)
  HD2018: keep NYC counties: 6,857 -> 183 rows (-6,674)


  HD2019: keep NYC counties: 6,559 -> 186 rows (-6,373)
  HD2020: keep NYC counties: 6,440 -> 185 rows (-6,255)
  HD2021: keep NYC counties: 6,289 -> 182 rows (-6,107)
  HD2022: keep NYC counties: 6,256 -> 181 rows (-6,075)
  HD2023: keep NYC counties: 6,163 -> 177 rows (-5,986)


  HD2024: keep NYC counties: 6,072 -> 173 rows (-5,899)
Code -3 (not applicable) cells mapped to NULL: {'SECTOR': 0, 'CONTROL': 7, 'ICLEVEL': 7}


year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
NYC colleges,191,191,187,183,186,185,182,181,177,173
CUNY units,21,21,21,21,21,21,21,21,21,21


  stg_ipeds__institutions: 1,836 rows x 13 cols, key ['year', 'unitid'] unique | local data/interim/staging/stg_ipeds__institutions.parquet | DB: 1836


## 2. IPEDS completions: NYC colleges, detailed majors

Revised file where it exists, else provisional (C2024, issue 2). Kept: rows whose college is in that year's NYC list. Dropped: `CIPCODE = 99` grand totals (issue 7), after checking they equal the sum of their detail. `AWLEVEL` is zero-padded until 2022 and not from 2023 (issue 5), so it's cast to int.

In [3]:
C_COLS = ["UNITID", "CIPCODE", "MAJORNUM", "AWLEVEL", "XCTOTALT", "CTOTALT", "CTOTALM", "CTOTALW"]
nyc_ids = inst.groupby("year").unitid.apply(set).to_dict()
c_parts, totals_ok = [], []
for f, path in zip(raw_files("ipeds_completions").file, raw_files("ipeds_completions").path):
    year = int(Path(f).stem[1:5])
    d, member, release = read_zip(path, prefer_revised=True, usecols=C_COLS)
    nyc = d[d.UNITID.isin(nyc_ids[year])].copy()
    log(f"C{year} ({release}): keep NYC colleges", len(d), len(nyc))
    for c in ["CTOTALT", "CTOTALM", "CTOTALW", "MAJORNUM", "AWLEVEL"]:
        nyc[c] = pd.to_numeric(nyc[c], errors="raise").astype("Int64")
    tot = nyc[nyc.CIPCODE == "99"]
    det = nyc[nyc.CIPCODE != "99"]
    cmp = tot.set_index(["UNITID", "MAJORNUM", "AWLEVEL"]).CTOTALT.sort_index()
    sums = det.groupby(["UNITID", "MAJORNUM", "AWLEVEL"]).CTOTALT.sum().reindex(cmp.index, fill_value=0)
    totals_ok.append({"year": year, "total rows": len(cmp), "equal to detail sum": int((cmp == sums).sum())})
    log(f"C{year}: drop CIPCODE 99 grand totals", len(nyc), len(det))
    c_parts.append(det.assign(year=year, release=release, source_file=f"{f}!{member}"))
display(pd.DataFrame(totals_ok).set_index("year").T)
assert all(r["total rows"] == r["equal to detail sum"] for r in totals_ok)

comp = pd.concat(c_parts, ignore_index=True)
assert comp.CIPCODE.str.fullmatch(r"\d\d\.\d{4}").all(), "non-detailed CIP left"
assert (comp.CTOTALM + comp.CTOTALW == comp.CTOTALT).all(), "men + women != total"
comp["cip_edition"] = comp.year.map(lambda y: "2010" if y <= 2019 else "2020")
print("Imputation flags:", comp.XCTOTALT.value_counts().to_dict())
stg_c = (comp.rename(columns={"UNITID": "unitid", "CIPCODE": "cip_code", "MAJORNUM": "major_number", "AWLEVEL": "award_level",
                              "CTOTALT": "awards_total", "CTOTALM": "awards_men", "CTOTALW": "awards_women",
                              "XCTOTALT": "imputation_flag"})
         [["year", "release", "unitid", "cip_code", "cip_edition", "major_number", "award_level", "awards_total",
           "awards_men", "awards_women", "imputation_flag", "source_file"]]
         .sort_values(["year", "unitid", "cip_code", "major_number", "award_level"]).reset_index(drop=True))
stg_c["year"] = stg_c.year.astype("Int64")

# Tie-out (card §5): CUNY first majors = CUNY's published Degrees Granted by Type; 2024: 11,402 associate's, 23,437 bachelor's
cuny = stg_c.merge(inst[["year", "unitid", "is_cuny"]], on=["year", "unitid"])
cuny = cuny[cuny.is_cuny & (cuny.major_number == 1) & cuny.award_level.isin([3, 5])]
t = cuny.pivot_table(index="year", columns="award_level", values="awards_total", aggfunc="sum").rename(columns={3: "associate", 5: "bachelor"})
display(t.T)
assert tuple(t.loc[2024]) == (11_402, 23_437)
summary.append(stage(stg_c, "stg_ipeds__completions", ["year", "unitid", "cip_code", "major_number", "award_level"],
                     expected_rows=58_343, source_key="ipeds_completions", notebook=NB, conn=conn,
                     comment="IPEDS completions, NYC colleges, detailed CIP (2010 to 2019, 2020 from 2020). 2024 provisional. Count awards on major_number = 1."))

  C2015 (revised): keep NYC colleges: 300,263 -> 5,826 rows (-294,437)
  C2015: drop CIPCODE 99 grand totals: 5,826 -> 5,337 rows (-489)


  C2016 (revised): keep NYC colleges: 302,043 -> 6,024 rows (-296,019)
  C2016: drop CIPCODE 99 grand totals: 6,024 -> 5,533 rows (-491)


  C2017 (revised): keep NYC colleges: 308,882 -> 6,405 rows (-302,477)
  C2017: drop CIPCODE 99 grand totals: 6,405 -> 5,911 rows (-494)


  C2018 (revised): keep NYC colleges: 290,149 -> 6,426 rows (-283,723)
  C2018: drop CIPCODE 99 grand totals: 6,426 -> 5,942 rows (-484)


  C2019 (revised): keep NYC colleges: 290,972 -> 6,493 rows (-284,479)
  C2019: drop CIPCODE 99 grand totals: 6,493 -> 5,984 rows (-509)


...

  C2024 (provisional): keep NYC colleges: 307,707 -> 6,455 rows (-301,252)
  C2024: drop CIPCODE 99 grand totals: 6,455 -> 5,941 rows (-514)


year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
total rows,489,491,494,484,509,525,512,527,517,514
equal to detail sum,489,491,494,484,509,525,512,527,517,514


Imputation flags: {'R': 58331, 'P': 8, 'Z': 4}


year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
award_level,,,,,,,,,,
associate,15478,16345,17372,18247,18880,18498,17596,14662,12246,11402
bachelor,22736,23709,24456,25371,27273,27219,29590,28008,25074,23437


  stg_ipeds__completions: 58,343 rows x 12 cols, key ['year', 'unitid', 'cip_code', 'major_number', 'award_level'] unique | local data/interim/staging/stg_ipeds__completions.parquet | DB: 58343


## 3. CUNY Programs: split into graduates and enrollment (issues 11–17)

Key `irp_code × college × award_name` (the IRP code alone isn't unique). Zeros are real and kept (closed or not-yet-open programs are told apart by `date_closed`). Every college and award name must be in the `ref/` maps (asserted).

In [4]:
pf = raw_files("cuny_programs").iloc[-1]
cp = pd.read_csv(pf.path, dtype=str, keep_default_na=False).apply(lambda s: s.str.strip())
print(f"CUNY Programs: {len(cp):,} rows")
units = pd.read_csv(REF / "cuny_unit_unitid.csv", dtype=str, keep_default_na=False)
awards = pd.read_csv(REF / "cuny_award_level.csv", dtype=str, keep_default_na=False)
assert not (missing := set(cp.COLLEGE) - set(units.college)), f"colleges not in ref/cuny_unit_unitid.csv: {missing}"
assert not (missing := set(cp["AWARD NAME"]) - set(awards.award_name)), f"awards not in ref/cuny_award_level.csv: {missing}"
print("ref award names not in the file (harmless):", sorted(set(awards.award_name) - set(cp["AWARD NAME"])) or "none")
assert set(units.unitid) <= set(inst[(inst.year == 2024) & inst.is_cuny].unitid), "a mapped UNITID isn't CUNY in HD 2024"

cp = cp.merge(units[["college", "unitid", "is_parent_assumption"]], left_on="COLLEGE", right_on="college", how="left")
cp = cp.merge(awards[["award_name", "award_level"]], left_on="AWARD NAME", right_on="award_name", how="left")
cp = cp.drop(columns=["college", "award_name"])   # keep the file's own spelling, renamed below
cp["unitid_is_parent_assumption"] = cp.is_parent_assumption.eq("true")
for c in ["DATE FIRST REGISTERED", "DATE CLOSED"]:
    blank = cp[c] == ""
    cp[c] = pd.to_datetime(cp[c].mask(blank), format="%m/%d/%Y", errors="raise")
print("Programs with a closing date:", cp.drop_duplicates(["IRP CODE", "COLLEGE", "AWARD NAME"])["DATE CLOSED"].notna().sum())
for c in ["TAP ELIGIBLE", "APTS ELIGIBLE", "VVTA ELIGIBLE"]:
    assert set(cp[c]) <= {"YES", "NO"}, set(cp[c])
    cp[c.lower().replace(" ", "_")] = cp[c].eq("YES")
cp["count"] = pd.to_numeric(cp.COUNT, errors="raise").astype("Int64")
cp["year"] = cp.PERIOD.str[-4:].astype(int).astype("Int64")   # AY_2023_2024 -> 2024 (year it ends); Fall_2025 -> 2025
cp["source_file"] = pf.file

COMMON = {"IRP CODE": "irp_code", "COLLEGE": "college", "PROGRAM NAME": "program_name", "AWARD NAME": "award_name",
          "AWARD NUMBER": "award_number", "CIP 2020 CODE": "cip_code", "DATE FIRST REGISTERED": "date_registered",
          "DATE CLOSED": "date_closed", "PERIOD": "period"}
KEEP = ["irp_code", "college", "unitid", "unitid_is_parent_assumption", "program_name", "award_name", "award_level",
        "award_number", "cip_code", "date_registered", "date_closed", "tap_eligible", "apts_eligible", "vvta_eligible",
        "period", "year"]
cp = cp.rename(columns=COMMON)
grad = cp[cp["RECORD TYPE"] == "Graduation"].rename(columns={"count": "graduates"})[KEEP + ["graduates", "source_file"]]
enr = cp[cp["RECORD TYPE"] == "Enrollment"].rename(columns={"count": "enrolled"})[KEEP + ["enrolled", "source_file"]]
log("split: Graduation rows", len(cp), len(grad)); log("split: Enrollment rows", len(cp), len(enr))
assert len(grad) + len(enr) == len(cp)
print("Graduation periods:", grad.period.min(), "->", grad.period.max(), "| Enrollment:", enr.period.min(), "->", enr.period.max())

# Tie-out (card §4): AY 2024-25 = 11,676 associate's, 22,591 bachelor's
g25 = grad[grad.year == 2025].groupby("award_level").graduates.sum()
print("AY 2024-25 graduates:", g25.to_dict()); assert (g25["associate"], g25["bachelor"]) == (11_676, 22_591)

KEY = ["irp_code", "college", "award_name", "period"]
summary.append(stage(grad.reset_index(drop=True), "stg_cuny__program_graduates", KEY, expected_rows=53_838,
                     source_key="cuny_programs", notebook=NB, conn=conn,
                     comment="CUNY graduates by program and academic year (year = year the AY ends). Counts a degree once per program: double majors twice."))
summary.append(stage(enr.reset_index(drop=True), "stg_cuny__program_enrollment", KEY, expected_rows=53_838,
                     source_key="cuny_programs", notebook=NB, conn=conn,
                     comment="CUNY fall enrollment by program."))

CUNY Programs: 107,676 rows
ref award names not in the file (harmless): none


Programs with a closing date: 476


  split: Graduation rows: 107,676 -> 53,838 rows (-53,838)
  split: Enrollment rows: 107,676 -> 53,838 rows (-53,838)
Graduation periods: AY_2007_2008 -> AY_2024_2025 | Enrollment: Fall_2008 -> Fall_2025
AY 2024-25 graduates: {'associate': 11676, 'bachelor': 22591, 'certificate': 439, 'graduate': 9983}


  stg_cuny__program_graduates: 53,838 rows x 18 cols, key ['irp_code', 'college', 'award_name', 'period'] unique | local data/interim/staging/stg_cuny__program_graduates.parquet | DB: 53838


  stg_cuny__program_enrollment: 53,838 rows x 18 cols, key ['irp_code', 'college', 'award_name', 'period'] unique | local data/interim/staging/stg_cuny__program_enrollment.parquet | DB: 53838


## 4. Open question from nb 05: Graduate Center bachelor's, CUNY Programs 78 below IPEDS (2024)

Graduate Center bachelor's (CUNY BA/BS, SPS) compared program by program, by CIP, between CUNY Programs (every unit mapped to UNITID 190576) and IPEDS first + second majors.

In [5]:
gc_cp = grad[(grad.unitid == "190576") & (grad.year == 2024) & (grad.award_level == "bachelor")]
gc_cp = gc_cp.groupby(["college", "cip_code"]).graduates.sum().rename("cuny_programs").reset_index()
gc_ip = stg_c[(stg_c.unitid == "190576") & (stg_c.year == 2024) & (stg_c.award_level == 5)]
gc_ip = gc_ip.groupby(["cip_code", "major_number"]).awards_total.sum().unstack(fill_value=0).add_prefix("ipeds_major_").reset_index()
gc = gc_cp.groupby("cip_code").cuny_programs.sum().reset_index().merge(gc_ip, on="cip_code", how="outer").fillna(0)
gc["ipeds_1st_plus_2nd"] = gc.filter(like="ipeds_major_").sum(axis=1)
gc["diff"] = gc.cuny_programs - gc.ipeds_1st_plus_2nd
print(f"Totals: CUNY Programs {gc.cuny_programs.sum():,.0f} | IPEDS 1st + 2nd {gc.ipeds_1st_plus_2nd.sum():,.0f} | diff {gc['diff'].sum():+,.0f}")
display(gc[gc["diff"] != 0].sort_values("diff"))
display(gc_cp.groupby("college").cuny_programs.sum().to_frame().T)

Totals: CUNY Programs 709 | IPEDS 1st + 2nd 787 | diff -78


,cip_code,cuny_programs,ipeds_major_1,ipeds_1st_plus_2nd,diff
164,51.2211,0,101,101,-101
110,40.0101,0,44,44,-44
85,24.0101,25,59,59,-34
158,51.0701,101,0,0,101


college,Labor and Urban Studies,School Prof Studies (GC)
cuny_programs,5,704


Finding (closes nb 05 issue 8). The −78 has two parts, neither a mapping error:
- IPEDS reports 78 Graduate Center bachelor's that CUNY Programs doesn't list: 44 in Physical Sciences (`40.0101`) and 34 more in Liberal Arts (`24.0101`). CUNY Programs has no Graduate Center program under those codes for 2024.
- **Same 101 graduates, different CIP:** health administration is `51.2211` in IPEDS and `51.0701` in CUNY Programs (nets to 0 in the total, but matters if the two sources are joined by major).

Effect: 0.3% of CUNY bachelor's. Treatment: none needed in staging. IPEDS stays the source for graduate counts (decided in nb 05); CUNY Programs is used for the latest year and enrollment. Core must not join the two sources on CIP for "same graduates" checks without this caveat.

## 5. Summary

In [6]:
display(pd.concat(summary, ignore_index=True))
if conn is not None:
    conn.close()

,table,rows,cols,key,source_key,notebook,in_db,loaded_utc
0,stg_ipeds__institutions,1836,13,year + unitid,ipeds_directory,23-load-graduates.ipynb,1836,2026-10-14T21:12:14+00:00
1,stg_ipeds__completions,58343,12,year + unitid + cip_code + major_number + award_level,ipeds_completions,23-load-graduates.ipynb,58343,2026-10-14T21:12:20+00:00
2,stg_cuny__program_graduates,53838,18,irp_code + college + award_name + period,cuny_programs,23-load-graduates.ipynb,53838,2026-10-14T21:12:21+00:00
3,stg_cuny__program_enrollment,53838,18,irp_code + college + award_name + period,cuny_programs,23-load-graduates.ipynb,53838,2026-10-14T21:12:22+00:00
